# Dataset Signal Analysis

## Data Preparation

In [1]:
from itertools import combinations
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import mutual_info_score
from sklearn.preprocessing import OrdinalEncoder

working_directory = Path.cwd().resolve()
PROJECT_ROOT = (
    working_directory
    if (working_directory / "app").exists()
    else working_directory.parent
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.ml.preprocessing import prepare_features_and_target

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "fraud_detection.csv"
TARGET = "Fraudulent"
CATEGORICAL_FEATURES = [
    "Transaction_Type",
    "Device_Used",
    "Location",
    "Payment_Method",
]
NUMERICAL_FEATURES = [
    "Transaction_Amount",
    "Time_of_Transaction",
    "Previous_Fraudulent_Transactions",
    "Account_Age",
    "Number_of_Transactions_Last_24H",
]

In [2]:
raw_data = pd.read_csv(DATA_PATH)
X, y = prepare_features_and_target(raw_data)
analysis_data = X.copy()
analysis_data[TARGET] = y

print(f"Raw shape: {raw_data.shape}")
print(f"Analysis shape after exact deduplication and identifier exclusion: {analysis_data.shape}")
print(f"Exact duplicates removed: {len(raw_data) - len(analysis_data):,}")
print("Predictive features:", X.columns.tolist())

Raw shape: (51000, 12)
Analysis shape after exact deduplication and identifier exclusion: (50119, 10)
Exact duplicates removed: 881
Predictive features: ['Transaction_Amount', 'Transaction_Type', 'Time_of_Transaction', 'Device_Used', 'Location', 'Previous_Fraudulent_Transactions', 'Account_Age', 'Number_of_Transactions_Last_24H', 'Payment_Method']


## Target Distribution

In [3]:
target_counts = y.value_counts().sort_index()
target_distribution = pd.DataFrame({
    "count": target_counts,
    "percentage": target_counts.div(len(y)).mul(100),
}).rename(index={0: "Legitimate (0)", 1: "Fraudulent (1)"})
display(target_distribution.style.format({"percentage": "{:.4f}%"}))
overall_fraud_rate = float(y.mean())
print(f"Overall fraud rate: {overall_fraud_rate:.4%}")

,count,percentage
Fraudulent,,
Legitimate (0),47652,95.0777%
Fraudulent (1),2467,4.9223%


Overall fraud rate: 4.9223%


## Categorical Fraud Rates

In [4]:
def categorical_fraud_summary(feature: str) -> pd.DataFrame:
    values = analysis_data[feature].astype("object").fillna("<MISSING>")
    summary = (
        pd.DataFrame({feature: values, TARGET: y})
        .groupby(feature, dropna=False)[TARGET]
        .agg(transaction_count="size", fraud_count="sum", fraud_rate="mean")
        .sort_values("fraud_rate", ascending=False)
    )
    summary["fraud_rate_percentage"] = summary["fraud_rate"].mul(100)
    return summary.drop(columns="fraud_rate")

categorical_summaries = {}
for feature in CATEGORICAL_FEATURES:
    categorical_summaries[feature] = categorical_fraud_summary(feature)
    print(feature)
    display(categorical_summaries[feature].style.format({"fraud_rate_percentage": "{:.4f}%"}))

Transaction_Type


,transaction_count,fraud_count,fraud_rate_percentage
Transaction_Type,,,
Online Purchase,9912,509,5.1352%
POS Payment,9957,499,5.0115%
Bill Payment,10161,506,4.9798%
Bank Transfer,10110,487,4.8170%
ATM Withdrawal,9979,466,4.6698%


Device_Used


,transaction_count,fraud_count,fraud_rate_percentage
Device_Used,,,
,2437,149,6.1141%
Mobile,15321,791,5.1628%
Unknown Device,1530,75,4.9020%
Desktop,15524,738,4.7539%
Tablet,15307,714,4.6645%


Location


,transaction_count,fraud_count,fraud_rate_percentage
Location,,,
Chicago,5968,329,5.5127%
Los Angeles,5910,305,5.1607%
Boston,6037,302,5.0025%
San Francisco,5881,294,4.9991%
Miami,5884,293,4.9796%
,2500,122,4.8800%
New York,6003,285,4.7476%
Houston,5935,277,4.6672%
Seattle,6001,260,4.3326%


Payment_Method


,transaction_count,fraud_count,fraud_rate_percentage
Payment_Method,,,
UPI,11671,596,5.1067%
Credit Card,11417,562,4.9225%
Debit Card,11614,571,4.9165%
Net Banking,11462,562,4.9032%
Invalid Method,1527,73,4.7806%
,2428,103,4.2422%


## Numerical Feature Comparison

In [5]:
numerical_by_target = analysis_data.groupby(TARGET)[NUMERICAL_FEATURES].agg(
    ["count", "mean", "median", "std"]
)
numerical_by_target.index = numerical_by_target.index.map(
    {0: "Legitimate (0)", 1: "Fraudulent (1)"}
)
display(numerical_by_target.T.style.format(precision=4))

## Previous Fraud History

In [6]:
previous_fraud_summary = (
    analysis_data.groupby("Previous_Fraudulent_Transactions")[TARGET]
    .agg(transaction_count="size", fraud_count="sum", fraud_rate="mean")
    .sort_index()
)
previous_fraud_summary["fraud_rate_percentage"] = previous_fraud_summary["fraud_rate"].mul(100)
display(previous_fraud_summary.drop(columns="fraud_rate").style.format({"fraud_rate_percentage": "{:.4f}%"}))

,transaction_count,fraud_count,fraud_rate_percentage
Previous_Fraudulent_Transactions,,,
0,10101,475,4.7025%
1,9956,507,5.0924%
2,10121,508,5.0193%
3,9902,494,4.9889%
4,10039,483,4.8112%


## Simple Feature Associations

In [7]:
numeric_associations = []
for feature in NUMERICAL_FEATURES:
    legitimate = analysis_data.loc[y == 0, feature].dropna()
    fraudulent = analysis_data.loc[y == 1, feature].dropna()
    pooled_standard_deviation = np.sqrt(
        ((len(legitimate) - 1) * legitimate.var() + (len(fraudulent) - 1) * fraudulent.var())
        / (len(legitimate) + len(fraudulent) - 2)
    )
    standardized_mean_difference = (
        (fraudulent.mean() - legitimate.mean()) / pooled_standard_deviation
        if pooled_standard_deviation > 0 else 0.0
    )
    numeric_associations.append({
        "Feature": feature,
        "Point_Biserial_Correlation": analysis_data[feature].corr(y),
        "Standardized_Mean_Difference": standardized_mean_difference,
    })
numeric_association_table = pd.DataFrame(numeric_associations)
numeric_association_table["Absolute_Correlation"] = numeric_association_table["Point_Biserial_Correlation"].abs()
numeric_association_table = numeric_association_table.sort_values("Absolute_Correlation", ascending=False)
display(numeric_association_table.style.format(precision=6))

,Feature,Point_Biserial_Correlation,Standardized_Mean_Difference,Absolute_Correlation
1,Time_of_Transaction,0.005854,0.027102,0.005854
0,Transaction_Amount,0.005800,0.026823,0.005800
3,Account_Age,0.005517,0.025503,0.005517
4,Number_of_Transactions_Last_24H,-0.003964,-0.018321,0.003964
2,Previous_Fraudulent_Transactions,0.000766,0.003542,0.000766


In [8]:
target_entropy = mutual_info_score(y, y)
categorical_associations = []
for feature in CATEGORICAL_FEATURES:
    values = analysis_data[feature].astype("object").fillna("<MISSING>")
    mutual_information = mutual_info_score(values, y)
    categorical_associations.append({
        "Feature": feature,
        "Mutual_Information": mutual_information,
        "MI_as_Fraction_of_Target_Entropy": mutual_information / target_entropy,
    })
categorical_association_table = (
    pd.DataFrame(categorical_associations)
    .sort_values("Mutual_Information", ascending=False)
)
display(categorical_association_table.style.format(precision=8))

,Feature,Mutual_Information,MI_as_Fraction_of_Target_Entropy
1,Device_Used,0.00011890,0.00060594
2,Location,0.00011048,0.00056304
3,Payment_Method,0.00003418,0.00017420
0,Transaction_Type,0.00002804,0.00014288


## Mutual Information Ranking

In [9]:
encoded_features = pd.DataFrame(index=X.index)
for feature in NUMERICAL_FEATURES:
    encoded_features[feature] = X[feature].fillna(X[feature].median())

categorical_values = X[CATEGORICAL_FEATURES].astype("object").fillna("<MISSING>")
ordinal_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
encoded_features[CATEGORICAL_FEATURES] = ordinal_encoder.fit_transform(categorical_values)
encoded_features = encoded_features[X.columns]
discrete_mask = [feature in CATEGORICAL_FEATURES for feature in encoded_features.columns]

observed_mi = mutual_info_classif(
    encoded_features, y, discrete_features=discrete_mask, random_state=42
)
rng = np.random.default_rng(42)
null_mi_scores = np.array([
    mutual_info_classif(
        encoded_features,
        rng.permutation(y.to_numpy()),
        discrete_features=discrete_mask,
        random_state=42 + permutation_number,
    )
    for permutation_number in range(20)
])

mutual_information_ranking = pd.DataFrame({
    "Feature": encoded_features.columns,
    "Observed_MI": observed_mi,
    "Null_MI_Mean": null_mi_scores.mean(axis=0),
    "Null_MI_95th_Percentile": np.quantile(null_mi_scores, 0.95, axis=0),
})
mutual_information_ranking["Above_Null_95th_Percentile"] = (
    mutual_information_ranking["Observed_MI"]
    > mutual_information_ranking["Null_MI_95th_Percentile"]
)
mutual_information_ranking = mutual_information_ranking.sort_values("Observed_MI", ascending=False)
display(mutual_information_ranking.style.format({
    "Observed_MI": "{:.8f}",
    "Null_MI_Mean": "{:.8f}",
    "Null_MI_95th_Percentile": "{:.8f}",
}))

,Feature,Observed_MI,Null_MI_Mean,Null_MI_95th_Percentile,Above_Null_95th_Percentile
5,Previous_Fraudulent_Transactions,0.00257439,0.00238714,0.00351531,False
7,Number_of_Transactions_Last_24H,0.00086683,0.00089715,0.00210534,False
3,Device_Used,0.00011890,0.00003885,0.00007868,True
4,Location,0.00011048,0.00008915,0.00014827,False
2,Time_of_Transaction,0.00009969,0.00073390,0.00214715,False
8,Payment_Method,0.00003418,0.00004021,0.00008167,False
1,Transaction_Type,0.00002804,0.00003105,0.00006771,False
0,Transaction_Amount,0.00000000,0.00037307,0.00136577,False
6,Account_Age,0.00000000,0.00043831,0.00153830,False


## Single-Feature and Pairwise Separation

In [10]:
numeric_rate_ranges = []
for feature in NUMERICAL_FEATURES:
    non_missing = analysis_data[feature].dropna()
    bins = pd.qcut(non_missing, q=4, duplicates="drop")
    fraud_rates = y.loc[non_missing.index].groupby(bins, observed=True).mean()
    numeric_rate_ranges.append({
        "Feature": feature,
        "Minimum_Bin_Fraud_Rate": fraud_rates.min(),
        "Maximum_Bin_Fraud_Rate": fraud_rates.max(),
        "Fraud_Rate_Range": fraud_rates.max() - fraud_rates.min(),
    })
numeric_rate_range_table = pd.DataFrame(numeric_rate_ranges).sort_values("Fraud_Rate_Range", ascending=False)
display(numeric_rate_range_table.style.format({
    "Minimum_Bin_Fraud_Rate": "{:.4%}",
    "Maximum_Bin_Fraud_Rate": "{:.4%}",
    "Fraud_Rate_Range": "{:.4%}",
}))

,Feature,Minimum_Bin_Fraud_Rate,Maximum_Bin_Fraud_Rate,Fraud_Rate_Range
4,Number_of_Transactions_Last_24H,4.6541%,5.2242%,0.5701%
3,Account_Age,4.7249%,5.1103%,0.3854%
1,Time_of_Transaction,4.7559%,5.1317%,0.3757%
0,Transaction_Amount,4.7267%,5.0630%,0.3362%
2,Previous_Fraudulent_Transactions,4.8112%,5.0193%,0.2080%


In [11]:
pairwise_summaries = []
pairwise_groups = []
minimum_pair_count = 500
for first_feature, second_feature in combinations(CATEGORICAL_FEATURES, 2):
    pair_data = analysis_data[[first_feature, second_feature, TARGET]].copy()
    pair_data[first_feature] = pair_data[first_feature].astype("object").fillna("<MISSING>")
    pair_data[second_feature] = pair_data[second_feature].astype("object").fillna("<MISSING>")
    grouped = (
        pair_data.groupby([first_feature, second_feature])[TARGET]
        .agg(transaction_count="size", fraud_count="sum", fraud_rate="mean")
        .reset_index()
    )
    eligible = grouped[grouped["transaction_count"] >= minimum_pair_count].copy()
    eligible["Feature_Pair"] = f"{first_feature} + {second_feature}"
    eligible["Absolute_Deviation_From_Overall"] = (eligible["fraud_rate"] - overall_fraud_rate).abs()
    pairwise_groups.append(eligible)
    pairwise_summaries.append({
        "Feature_Pair": f"{first_feature} + {second_feature}",
        "Eligible_Groups": len(eligible),
        "Minimum_Fraud_Rate": eligible["fraud_rate"].min(),
        "Maximum_Fraud_Rate": eligible["fraud_rate"].max(),
        "Fraud_Rate_Range": eligible["fraud_rate"].max() - eligible["fraud_rate"].min(),
    })
pairwise_summary_table = pd.DataFrame(pairwise_summaries).sort_values("Fraud_Rate_Range", ascending=False)
display(pairwise_summary_table.style.format({
    "Minimum_Fraud_Rate": "{:.4%}",
    "Maximum_Fraud_Rate": "{:.4%}",
    "Fraud_Rate_Range": "{:.4%}",
}))

largest_pairwise_deviations = (
    pd.concat(pairwise_groups, ignore_index=True)
    .sort_values("Absolute_Deviation_From_Overall", ascending=False)
    .head(10)
)
display(largest_pairwise_deviations.style.format({
    "fraud_rate": "{:.4%}",
    "Absolute_Deviation_From_Overall": "{:.4%}",
}))

,Feature_Pair,Eligible_Groups,Minimum_Fraud_Rate,Maximum_Fraud_Rate,Fraud_Rate_Range
4,Device_Used + Payment_Method,20,3.4530%,9.2334%,5.7804%
3,Device_Used + Location,27,3.1772%,6.2011%,3.0239%
1,Transaction_Type + Location,42,3.6565%,6.4701%,2.8136%
5,Location + Payment_Method,36,3.3733%,5.8492%,2.4759%
0,Transaction_Type + Device_Used,16,4.3464%,6.6667%,2.3202%
2,Transaction_Type + Payment_Method,21,3.6398%,5.5312%,1.8913%


,Transaction_Type,Device_Used,transaction_count,fraud_count,fraud_rate,Feature_Pair,Absolute_Deviation_From_Overall,Location,Payment_Method
109,nan,,574,53,9.2334%,Device_Used + Payment_Method,4.3112%,nan,UPI
105,nan,Tablet,1857,59,3.1772%,Device_Used + Location,1.7451%,Seattle,nan
6,Bill Payment,,540,36,6.6667%,Transaction_Type + Device_Used,1.7444%,nan,nan
139,nan,nan,1334,45,3.3733%,Location + Payment_Method,1.5490%,Houston,Debit Card
35,Bill Payment,nan,1221,79,6.4701%,Transaction_Type + Location,1.5478%,Chicago,nan
121,nan,Tablet,724,25,3.4530%,Device_Used + Payment_Method,1.4692%,nan,
43,Online Purchase,nan,1207,77,6.3795%,Transaction_Type + Location,1.4572%,Chicago,nan
66,Bill Payment,nan,522,19,3.6398%,Transaction_Type + Payment_Method,1.2824%,nan,
95,nan,Mobile,1790,111,6.2011%,Device_Used + Location,1.2788%,San Francisco,nan
159,nan,nan,1397,51,3.6507%,Location + Payment_Method,1.2716%,Seattle,Debit Card


## Dataset Signal Conclusion

In [12]:
max_absolute_correlation = float(numeric_association_table["Absolute_Correlation"].max())
max_categorical_entropy_fraction = float(
    categorical_association_table["MI_as_Fraction_of_Target_Entropy"].max()
)
max_observed_mi = float(mutual_information_ranking["Observed_MI"].max())
features_above_null = int(mutual_information_ranking["Above_Null_95th_Percentile"].sum())
max_numeric_rate_range = float(numeric_rate_range_table["Fraud_Rate_Range"].max())
max_pairwise_deviation = float(largest_pairwise_deviations["Absolute_Deviation_From_Overall"].max())

meaningful_signal = (
    max_absolute_correlation >= 0.10
    or max_categorical_entropy_fraction >= 0.02
    or (max_observed_mi >= 0.01 and features_above_null > 0)
)
weak_signal = (
    features_above_null > 0
    or max_absolute_correlation >= 0.03
    or max_categorical_entropy_fraction >= 0.01
)
if meaningful_signal:
    signal_statement = "At least one diagnostic indicates potentially meaningful fraud signal, although it still requires validation."
elif weak_signal:
    signal_statement = "Only weak feature-target signals are visible; none meets the notebook's meaningful-effect thresholds."
else:
    signal_statement = "The target appears largely independent of the provided predictors in these diagnostics."

display(Markdown(
    f"{signal_statement}\n\n"
    f"- Maximum absolute numerical correlation: **{max_absolute_correlation:.6f}**.\n"
    f"- Maximum categorical MI as a fraction of target entropy: **{max_categorical_entropy_fraction:.6f}**.\n"
    f"- Maximum all-feature mutual information: **{max_observed_mi:.6f}**; "
    f"**{features_above_null}** of {len(mutual_information_ranking)} features exceeded their permutation-based 95th-percentile noise reference.\n"
    f"- Largest numerical quartile fraud-rate range: **{max_numeric_rate_range:.4%}**.\n"
    f"- Largest fraud-rate deviation among categorical pairs with at least {minimum_pair_count} records: **{max_pairwise_deviation:.4%}**.\n\n"
    "These diagnostics show association, not causation. Small subgroup differences and isolated permutation exceedances can occur by chance, especially when many features and groups are examined. No labels were altered and no classifier was trained."
))

Only weak feature-target signals are visible; none meets the notebook's meaningful-effect thresholds.

- Maximum absolute numerical correlation: **0.005854**.
- Maximum categorical MI as a fraction of target entropy: **0.000606**.
- Maximum all-feature mutual information: **0.002574**; **1** of 9 features exceeded their permutation-based 95th-percentile noise reference.
- Largest numerical quartile fraud-rate range: **0.5701%**.
- Largest fraud-rate deviation among categorical pairs with at least 500 records: **4.3112%**.

These diagnostics show association, not causation. Small subgroup differences and isolated permutation exceedances can occur by chance, especially when many features and groups are examined. No labels were altered and no classifier was trained.